# ARC-v0.23 — Eq. 6 Common-State Operator Replay

**Purpose.** Empirically decompose the cross-fidelity next-state difference from the manuscript

\[
\delta_{t+1}
=
[T_H(q_H^t)-T_H(q_L^t)]
+
[T_H(q_L^t)-T_L(q_L^t)]
\]

into:

- **propagated-state component**: \(T_H(q_H^t)-T_H(q_L^t)\)
- **direct-fidelity component**: \(T_H(q_L^t)-T_L(q_L^t)\)

This notebook is a **frozen falsification audit**. It does **not** assume that representation approximation must show larger direct perturbations, or that search-effort approximation must show stronger cancellation. The goal is to measure the components and test whether their round profiles differ across approximation interventions.

### Primary scope

Same encoder/corpus/policies/split, to isolate the approximation intervention as much as possible:

1. **E5 representation:** IVF-PQ32 → IVF-SQ8, nprobe=64
2. **E5 IVF search effort:** SQ8 nprobe 8 → 64
3. **E5 HNSW search effort:** efSearch 8 → 256

Primary evaluation uses the existing **FEVER FIT-held-out validation split** only.

### Important interpretation rule

This is post-primary mechanism hardening. It must **not** be described as preregistered evidence from the original paper lineage. If the component profiles do not cleanly separate approximation classes, retain that null/ambiguous result and weaken the mechanism language accordingly.

In [ ]:
# Cell 1 — Install/import dependencies
!pip -q install faiss-cpu pyarrow tqdm

from pathlib import Path
from datetime import datetime, timezone
from collections import defaultdict
import hashlib, json, math, os, random, time, warnings, gc

import faiss
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from google.colab import drive

warnings.filterwarnings("ignore", category=FutureWarning)

SEED = 20260823
random.seed(SEED)
np.random.seed(SEED)

DIM = 384
N_DOCS = 5_416_568
N_DEV = 6_666
TOP_RETRIEVE = 100
UTILITY_K = 10
ROUNDS = 4

ALPHAS = [0.1, 0.3, 0.5, 0.7]
MEAN_K = [5, 20, 50]
SOFTMAX_K = [5, 20]
TEMPERATURES = [0.05, 0.1, 0.2, 0.5]

BOOTSTRAP_REPS = 10_000

# Execution controls.
# Set FULL_RUN=False first to smoke-test 25 validation queries.
FULL_RUN = True
SMOKE_N_QUERIES = 25
CHECKPOINT_EVERY_QUERIES = 10

faiss.omp_set_num_threads(os.cpu_count() or 1)

print("faiss:", faiss.__version__)
print("threads:", faiss.omp_get_max_threads())

In [ ]:
# Cell 2 — Mount Drive and resolve frozen source artifacts
DRIVE_ROOT = Path("/content/drive/MyDrive")
if not DRIVE_ROOT.is_dir():
    drive.mount("/content/drive")

ARC_ROOT = DRIVE_ROOT / "rag-pq-checkpoints" / "arc-v0"

# ---------- v0.18 E5 representation source ----------
V018_ROOT = ARC_ROOT / "cross-encoder-fever-replication-v018"
V018_PREFERRED = V018_ROOT / "20260819-015645"

def complete_v018(p):
    return (
        p.is_dir()
        and (p / "dev_query_embeddings.float32.npy").is_file()
        and (p / "dev_query_ids.txt").is_file()
        and (p / "corpus_embeddings.float16.memmap").is_file()
        and (p / "v018_cross_encoder_protocol.json").is_file()
    )

if complete_v018(V018_PREFERRED):
    V018_RUN = V018_PREFERRED
else:
    candidates = sorted([p for p in V018_ROOT.iterdir() if complete_v018(p)], reverse=True)
    if not candidates:
        raise FileNotFoundError("No complete ARC-v0.18 E5 run found.")
    V018_RUN = candidates[0]

QUERY_EMB_PATH = V018_RUN / "dev_query_embeddings.float32.npy"
QUERY_IDS_PATH = V018_RUN / "dev_query_ids.txt"
CORPUS_MEMMAP_PATH = V018_RUN / "corpus_embeddings.float16.memmap"
V018_PROTOCOL = V018_RUN / "v018_cross_encoder_protocol.json"

PQ32_PATH = V018_RUN / "fever-e5-small-v2-ivfpq-nlist4096-m32-nbits8.faiss"
SQ8_PATH = V018_RUN / "fever-e5-small-v2-ivfsq8-nlist4096.faiss"

# ---------- v0.13 authoritative split ----------
V013_SPLIT_CANDIDATES = [
    ARC_ROOT / "fever-boundary-external-replication-v013" / "20260817-151852" / "v013_boundary_query_split.csv",
    ARC_ROOT / "fever-boundary-external-replication-v013" / "20260817-140640" / "v013_boundary_query_split.csv",
]
V013_SPLIT = next((p for p in V013_SPLIT_CANDIDATES if p.is_file()), None)
if V013_SPLIT is None:
    raise FileNotFoundError("Could not resolve v013_boundary_query_split.csv")

# ---------- HNSW source ----------
HNSW_ROOT = ARC_ROOT / "hnsw-mechanism-replication-v020c"
HNSW_INDEX = HNSW_ROOT / "fever-e5-small-v2-hnswflat-m32-efc200.faiss"
HNSW_FREEZE = HNSW_ROOT / "V020C_FROZEN_CONTRAST.json"

# ---------- audited qrels in corpus-row space ----------
SEALED_QRELS = (
    DRIVE_ROOT / "hc-rars-fever-5m-untouched-confirmation-v1"
    / "stage2" / "dev_qrels_rows.csv"
)

required = [
    QUERY_EMB_PATH, QUERY_IDS_PATH, CORPUS_MEMMAP_PATH, V018_PROTOCOL,
    PQ32_PATH, SQ8_PATH, V013_SPLIT, HNSW_INDEX, HNSW_FREEZE, SEALED_QRELS
]
missing = [str(p) for p in required if not p.is_file()]
if missing:
    raise FileNotFoundError("Missing required artifacts:\n" + "\n".join(missing))

print("V018_RUN:", V018_RUN)
print("SPLIT:", V013_SPLIT)
print("HNSW:", HNSW_INDEX)
print("PATH AUDIT: PASS")

In [ ]:
# Cell 3 — Hash helpers and freeze protocol BEFORE replay outcomes
def sha256_file(path, chunk=16*1024*1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
OUT_ROOT = ARC_ROOT / "eq6-common-state-operator-replay-v023"
OUT = OUT_ROOT / RUN_ID
OUT.mkdir(parents=True, exist_ok=False)

hnsw_freeze = json.loads(HNSW_FREEZE.read_text())
LOW_EF = int(hnsw_freeze["low_efSearch"])
HIGH_EF = int(hnsw_freeze["high_efSearch"])

assert LOW_EF == 8 and HIGH_EF == 256, (LOW_EF, HIGH_EF)

PROTOCOL = {
    "status": "ARC_V023_EQ6_COMMON_STATE_PROTOCOL_FROZEN_BEFORE_REPLAY_OUTCOMES",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "seed": SEED,
    "scope": "FEVER E5 FIT-held-out validation",
    "source_v018_run": str(V018_RUN),
    "source_v018_protocol_sha256": sha256_file(V018_PROTOCOL),
    "source_v013_split_sha256": sha256_file(V013_SPLIT),
    "source_hnsw_index_sha256": sha256_file(HNSW_INDEX),
    "source_hnsw_freeze_sha256": sha256_file(HNSW_FREEZE),
    "interventions": {
        "e5_representation": {
            "low": "IVF-PQ32 nprobe=64",
            "high": "IVF-SQ8 nprobe=64"
        },
        "e5_nprobe": {
            "low": "IVF-SQ8 nprobe=8",
            "high": "IVF-SQ8 nprobe=64"
        },
        "e5_hnsw": {
            "low": f"HNSW efSearch={LOW_EF}",
            "high": f"HNSW efSearch={HIGH_EF}"
        },
    },
    "rounds": ROUNDS,
    "top_retrieve": TOP_RETRIEVE,
    "utility_k": UTILITY_K,
    "policy_grid": {
        "alphas": ALPHAS,
        "mean_k": MEAN_K,
        "softmax_k": SOFTMAX_K,
        "temperatures": TEMPERATURES,
        "count": 44,
    },
    "components": {
        "propagation": "T_H(q_H^t) - T_H(q_L^t)",
        "direct": "T_H(q_L^t) - T_L(q_L^t)",
        "total": "T_H(q_H^t) - T_L(q_L^t)",
    },
    "primary_questions": [
        "Are direct and propagated state components separately non-zero/measurable?",
        "Do component norms, alignment, and cancellation profiles differ across interventions?",
        "Do fixed-common-state candidate, feedback, and utility perturbations differ across interventions?",
    ],
    "falsification_rule": (
        "No mechanism class separation is assumed. Stronger mechanism language is allowed only "
        "if query-cluster uncertainty supports a consistent distinction; null/ambiguous outcomes are retained."
    ),
    "statistical_unit": "query; all policy/round realizations travel with the sampled query",
    "bootstrap_reps": BOOTSTRAP_REPS,
    "test_accessed": False,
    "test_relevance_accessed": False,
}

PROTOCOL_PATH = OUT / "v023_eq6_common_state_protocol.json"
PROTOCOL_PATH.write_text(json.dumps(PROTOCOL, indent=2, sort_keys=True))
PROTOCOL_SHA = sha256_file(PROTOCOL_PATH)

(OUT / "V023_PROTOCOL_SHA256.txt").write_text(
    f"{PROTOCOL_SHA}  {PROTOCOL_PATH.name}\n"
)

print("OUTPUT:", OUT)
print("PROTOCOL SHA:", PROTOCOL_SHA)
print("PROTOCOL FROZEN — PASS")

In [ ]:
# Cell 4 — Load split, queries, qrels, corpus vectors, and indexes
split = pd.read_csv(V013_SPLIT)

# Be tolerant to naming used by prior notebooks.
qid_col = next(c for c in ["query_id", "query-id", "qid"] if c in split.columns)
split_col = next(c for c in ["split", "partition", "role"] if c in split.columns)

split[qid_col] = split[qid_col].astype(str)
split[split_col] = split[split_col].astype(str).str.lower()

VAL_IDS = set(split.loc[
    split[split_col].isin(["validation", "val", "heldout", "held-out"]),
    qid_col
].astype(str))

if len(VAL_IDS) != 3316:
    print("WARNING: expected 3316 validation ids, found", len(VAL_IDS))

DEV_QUERY_IDS = QUERY_IDS_PATH.read_text().splitlines()
queries = np.load(QUERY_EMB_PATH).astype(np.float32)
assert queries.shape == (N_DEV, DIM)

qid_to_row = {str(q): i for i, q in enumerate(DEV_QUERY_IDS)}
VAL_IDS = [q for q in DEV_QUERY_IDS if q in VAL_IDS]
assert VAL_IDS, "No validation IDs matched query embeddings."

expected_bytes = N_DOCS * DIM * np.dtype(np.float16).itemsize
assert CORPUS_MEMMAP_PATH.stat().st_size == expected_bytes

corpus_mm = np.memmap(
    CORPUS_MEMMAP_PATH,
    dtype=np.float16,
    mode="r",
    shape=(N_DOCS, DIM),
)

qr = pd.read_csv(SEALED_QRELS)
assert {"query-id","corpus-row","score"}.issubset(qr.columns)

QRELS = defaultdict(set)
for _, r in qr.iterrows():
    qid = str(r["query-id"])
    if qid in qid_to_row and float(r["score"]) > 0:
        QRELS[qid].add(int(r["corpus-row"]))

missing_qrels = [q for q in VAL_IDS if not QRELS[q]]
assert not missing_qrels, f"Missing qrels for {len(missing_qrels)} validation queries"

pq32 = faiss.read_index(str(PQ32_PATH))
sq8 = faiss.read_index(str(SQ8_PATH))
hnsw = faiss.read_index(str(HNSW_INDEX))

assert pq32.ntotal == sq8.ntotal == hnsw.ntotal == N_DOCS

print("validation queries:", len(VAL_IDS))
print("indexes loaded:", pq32.ntotal, sq8.ntotal, hnsw.ntotal)
print("DATA/INDEX LOAD — PASS")

In [ ]:
# Cell 5 — Frozen 44-policy grid
POLICIES = []

for alpha in ALPHAS:
    for k in MEAN_K:
        POLICIES.append({
            "family": "mean",
            "alpha": float(alpha),
            "k": int(k),
            "temperature": np.nan,
            "config_key": f"mean|a={alpha}|k={k}"
        })

    for k in SOFTMAX_K:
        for tau in TEMPERATURES:
            POLICIES.append({
                "family": "softmax",
                "alpha": float(alpha),
                "k": int(k),
                "temperature": float(tau),
                "config_key": f"softmax|a={alpha}|k={k}|tau={tau}"
            })

assert len(POLICIES) == 44
pd.DataFrame(POLICIES).to_csv(OUT / "v023_frozen_policy_grid.csv", index=False)

display(pd.DataFrame(POLICIES).head())
print("POLICY GRID — PASS")

In [ ]:
# Cell 6 — Numeric/retrieval/utility helpers
def norm_vec(x, eps=1e-12):
    x = np.asarray(x, dtype=np.float32)
    return x / max(float(np.linalg.norm(x)), eps)

def cosine_distance(a, b):
    a = norm_vec(a); b = norm_vec(b)
    return 1.0 - float(np.dot(a, b))

def jaccard_distance(a, b):
    A = set(map(int, a))
    B = set(map(int, b))
    return 1.0 - len(A & B) / max(1, len(A | B))

def ndcg(ids, relevant, k=UTILITY_K):
    ids = np.asarray(ids, dtype=np.int64)[:k]
    discounts = 1.0 / np.log2(np.arange(2, len(ids)+2))
    gains = np.asarray([1.0 if int(i) in relevant else 0.0 for i in ids])
    dcg = float(np.sum(gains * discounts))
    m = min(k, len(relevant))
    if m == 0:
        return 0.0
    idcg = float(np.sum(1.0 / np.log2(np.arange(2, m+2))))
    return dcg / idcg

def fetch_docs(ids):
    ids = np.asarray(ids, dtype=np.int64)
    x = np.asarray(corpus_mm[ids], dtype=np.float32)
    x /= np.maximum(np.linalg.norm(x, axis=1, keepdims=True), 1e-12)
    return x

def feedback_vector(scores, ids, policy):
    k = int(policy["k"])
    ids = np.asarray(ids[:k], dtype=np.int64)
    scores = np.asarray(scores[:k], dtype=np.float64)
    docs = fetch_docs(ids)

    if policy["family"] == "mean":
        f = docs.mean(axis=0)
    else:
        tau = float(policy["temperature"])
        z = scores / tau
        z -= z.max()
        w = np.exp(z)
        w /= w.sum()
        f = (docs * w[:, None]).sum(axis=0)

    return norm_vec(f)

def update_state(q0, feedback, alpha):
    return norm_vec((1.0-float(alpha))*q0 + float(alpha)*feedback)

def component_alignment(v_prop, v_direct, eps=1e-12):
    np_ = float(np.linalg.norm(v_prop))
    nd_ = float(np.linalg.norm(v_direct))
    if np_ < eps or nd_ < eps:
        return np.nan
    return float(np.dot(v_prop, v_direct) / (np_ * nd_))

print("HELPERS — READY")

In [ ]:
# Cell 7 — Unified low/high search abstraction
MECHANISMS = ["e5_representation", "e5_nprobe", "e5_hnsw"]

def search_mech(mechanism, fidelity, q):
    q = norm_vec(q)[None, :]

    if mechanism == "e5_representation":
        if fidelity == "L":
            pq32.nprobe = 64
            s, I = pq32.search(q, TOP_RETRIEVE)
        else:
            sq8.nprobe = 64
            s, I = sq8.search(q, TOP_RETRIEVE)

    elif mechanism == "e5_nprobe":
        if fidelity == "L":
            sq8.nprobe = 8
        else:
            sq8.nprobe = 64
        s, I = sq8.search(q, TOP_RETRIEVE)

    elif mechanism == "e5_hnsw":
        hnsw.hnsw.efSearch = LOW_EF if fidelity == "L" else HIGH_EF
        s, I = hnsw.search(q, TOP_RETRIEVE)

    else:
        raise ValueError(mechanism)

    s = s[0]
    I = I[0]
    valid = I >= 0
    return s[valid], I[valid]

print("SEARCH ABSTRACTION — READY")

## What is measured at each transition

At round \(t\), we have actual branch states \(q_L^t,q_H^t\). We evaluate three operator applications:

\[
z_{HH}=T_H(q_H^t),\quad
z_{HL}=T_H(q_L^t),\quad
z_{LL}=T_L(q_L^t).
\]

Then:

\[
v_{\text{prop}}=z_{HH}-z_{HL},\qquad
v_{\text{direct}}=z_{HL}-z_{LL},
\]

and exactly:

\[
z_{HH}-z_{LL}
=
v_{\text{prop}}+v_{\text{direct}}.
\]

The notebook reports Euclidean component norms, component alignment, and

\[
\text{cancellation ratio}
=
\frac{\|v_{\text{prop}}+v_{\text{direct}}\|}
{\|v_{\text{prop}}\|+\|v_{\text{direct}}\|}.
\]

A ratio near 1 means components reinforce or do not cancel much; a smaller value indicates stronger vector cancellation. This is descriptive unless uncertainty supports a consistent intervention difference.

In [ ]:
# Cell 8 — One query × policy × mechanism common-state replay
def replay_one(qid, policy, mechanism):
    q0 = norm_vec(queries[qid_to_row[qid]])
    qL = q0.copy()
    qH = q0.copy()
    rel = QRELS[qid]

    rows = []

    for t in range(ROUNDS):
        # Actual low branch L(qL)
        sL_qL, iL_qL = search_mech(mechanism, "L", qL)

        # Actual high branch H(qH)
        sH_qH, iH_qH = search_mech(mechanism, "H", qH)

        # Common-state counterfactual H(qL)
        sH_qL, iH_qL = search_mech(mechanism, "H", qL)

        k = int(policy["k"])
        if min(len(iL_qL), len(iH_qH), len(iH_qL)) < k:
            raise RuntimeError(f"Fewer than k valid results: {qid} {policy} {mechanism}")

        # Feedback operators
        fL_qL = feedback_vector(sL_qL, iL_qL, policy)
        fH_qH = feedback_vector(sH_qH, iH_qH, policy)
        fH_qL = feedback_vector(sH_qL, iH_qL, policy)

        zLL = update_state(q0, fL_qL, policy["alpha"])
        zHH = update_state(q0, fH_qH, policy["alpha"])
        zHL = update_state(q0, fH_qL, policy["alpha"])

        v_prop = zHH - zHL
        v_direct = zHL - zLL
        v_total = zHH - zLL

        prop_norm = float(np.linalg.norm(v_prop))
        direct_norm = float(np.linalg.norm(v_direct))
        total_norm = float(np.linalg.norm(v_total))

        denom = prop_norm + direct_norm
        cancellation_ratio = total_norm / denom if denom > 1e-12 else np.nan

        # Candidate decomposition proxies
        cand_direct = jaccard_distance(iH_qL[:TOP_RETRIEVE], iL_qL[:TOP_RETRIEVE])
        cand_prop = jaccard_distance(iH_qH[:TOP_RETRIEVE], iH_qL[:TOP_RETRIEVE])
        cand_total = jaccard_distance(iH_qH[:TOP_RETRIEVE], iL_qL[:TOP_RETRIEVE])

        # Feedback-vector distances
        fb_direct = cosine_distance(fH_qL, fL_qL)
        fb_prop = cosine_distance(fH_qH, fH_qL)
        fb_total = cosine_distance(fH_qH, fL_qL)

        # Utility decomposition under common-state and actual high state
        uL_qL = ndcg(iL_qL, rel)
        uH_qL = ndcg(iH_qL, rel)
        uH_qH = ndcg(iH_qH, rel)

        util_direct_signed = uH_qL - uL_qL
        util_prop_signed = uH_qH - uH_qL
        util_total_signed = uH_qH - uL_qL

        # Numerical identity checks
        identity_err = float(np.linalg.norm(v_total - (v_prop + v_direct)))
        utility_identity_err = abs(util_total_signed - (util_prop_signed + util_direct_signed))

        rows.append({
            "query_id": str(qid),
            "mechanism": mechanism,
            "round": int(t),
            "family": policy["family"],
            "alpha": float(policy["alpha"]),
            "k": int(policy["k"]),
            "temperature": (
                float(policy["temperature"])
                if not pd.isna(policy["temperature"]) else np.nan
            ),
            "config_key": policy["config_key"],

            "state_prop_norm": prop_norm,
            "state_direct_norm": direct_norm,
            "state_total_norm": total_norm,
            "state_component_cosine": component_alignment(v_prop, v_direct),
            "state_cancellation_ratio": cancellation_ratio,

            "state_prop_cosdist": cosine_distance(zHH, zHL),
            "state_direct_cosdist": cosine_distance(zHL, zLL),
            "state_total_cosdist": cosine_distance(zHH, zLL),

            "candidate_prop_jaccard": cand_prop,
            "candidate_direct_jaccard": cand_direct,
            "candidate_total_jaccard": cand_total,

            "feedback_prop_cosdist": fb_prop,
            "feedback_direct_cosdist": fb_direct,
            "feedback_total_cosdist": fb_total,

            "utility_L_qL": uL_qL,
            "utility_H_qL": uH_qL,
            "utility_H_qH": uH_qH,
            "utility_direct_signed": util_direct_signed,
            "utility_prop_signed": util_prop_signed,
            "utility_total_signed": util_total_signed,
            "utility_direct_abs": abs(util_direct_signed),
            "utility_prop_abs": abs(util_prop_signed),
            "utility_total_abs": abs(util_total_signed),

            "eq6_vector_identity_error": identity_err,
            "utility_additive_identity_error": utility_identity_err,
        })

        # Advance actual paired trajectories.
        qL = zLL
        qH = zHH

    return rows

print("REPLAY FUNCTION — READY")

In [ ]:
# Cell 9 — Smoke test one query/policy/mechanism and verify identities
smoke_qid = VAL_IDS[0]
smoke_policy = POLICIES[0]
smoke = pd.DataFrame(replay_one(smoke_qid, smoke_policy, "e5_representation"))

display(smoke[[
    "round",
    "state_prop_norm","state_direct_norm","state_total_norm",
    "state_component_cosine","state_cancellation_ratio",
    "eq6_vector_identity_error","utility_additive_identity_error"
]])

assert smoke["eq6_vector_identity_error"].max() < 1e-6
assert smoke["utility_additive_identity_error"].max() < 1e-10

print("EQ.6 IDENTITY SMOKE TEST — PASS")

In [ ]:
# Cell 10 — Resumable validation replay
selected_ids = VAL_IDS if FULL_RUN else VAL_IDS[:SMOKE_N_QUERIES]
mode = "full" if FULL_RUN else f"smoke-{len(selected_ids)}"

RUN_DIR = OUT / mode
RUN_DIR.mkdir(parents=True, exist_ok=True)

print("mode:", mode)
print("queries:", len(selected_ids))
print("expected rows:", len(selected_ids) * len(POLICIES) * len(MECHANISMS) * ROUNDS)

for start in range(0, len(selected_ids), CHECKPOINT_EVERY_QUERIES):
    stop = min(start + CHECKPOINT_EVERY_QUERIES, len(selected_ids))
    cp = RUN_DIR / f"replay_{start:04d}_{stop:04d}.parquet"

    if cp.exists():
        print("skip", cp.name)
        continue

    rows = []
    t0 = time.perf_counter()

    for qi in range(start, stop):
        qid = selected_ids[qi]
        for mechanism in MECHANISMS:
            for policy in POLICIES:
                rows.extend(replay_one(qid, policy, mechanism))

    df_cp = pd.DataFrame(rows)

    tmp = cp.with_suffix(".tmp.parquet")
    df_cp.to_parquet(tmp, index=False)
    os.replace(tmp, cp)

    dt = time.perf_counter() - t0
    print(f"wrote {cp.name}: {len(df_cp):,} rows | {dt:.1f}s")

parts = sorted(RUN_DIR.glob("replay_*.parquet"))
assert parts, "No replay checkpoints produced."

replay = pd.concat([pd.read_parquet(p) for p in parts], ignore_index=True)

expected = len(selected_ids) * 44 * 3 * ROUNDS
assert len(replay) == expected, (len(replay), expected)
assert replay["query_id"].nunique() == len(selected_ids)

MERGED = OUT / f"v023_{mode}_common_state_replay.parquet"
replay.to_parquet(MERGED, index=False)

print("merged:", MERGED)
print("rows:", len(replay))
print("REPLAY — COMPLETE")

In [ ]:
# Cell 11 — Integrity audits
assert replay["eq6_vector_identity_error"].max() < 1e-5
assert replay["utility_additive_identity_error"].max() < 1e-10

# Cancellation ratio should be in [0,1] up to tiny numerical tolerance.
cr = replay["state_cancellation_ratio"].dropna()
assert (cr >= -1e-6).all()
assert (cr <= 1.0 + 1e-5).all()

print("max vector identity error:", replay["eq6_vector_identity_error"].max())
print("max utility identity error:", replay["utility_additive_identity_error"].max())
print("cancellation ratio range:", (cr.min(), cr.max()))
print("INTEGRITY AUDIT — PASS")

In [ ]:
# Cell 12 — Query-level aggregation (query is the sampling unit)
MEASURES = [
    "state_prop_norm",
    "state_direct_norm",
    "state_total_norm",
    "state_component_cosine",
    "state_cancellation_ratio",
    "state_prop_cosdist",
    "state_direct_cosdist",
    "candidate_prop_jaccard",
    "candidate_direct_jaccard",
    "feedback_prop_cosdist",
    "feedback_direct_cosdist",
    "utility_prop_abs",
    "utility_direct_abs",
    "utility_prop_signed",
    "utility_direct_signed",
]

# Average policies and rounds within each query first.
query_mech = (
    replay.groupby(["query_id","mechanism"], as_index=False)[MEASURES]
    .mean()
)

# Also retain round-specific query summaries.
query_mech_round = (
    replay.groupby(["query_id","mechanism","round"], as_index=False)[MEASURES]
    .mean()
)

display(query_mech.groupby("mechanism")[MEASURES].mean().round(6))

In [ ]:
# Cell 13 — Query-cluster bootstrap CIs
rng = np.random.default_rng(SEED + 23)

def bootstrap_mean_ci(df, value_col, reps=BOOTSTRAP_REPS):
    x = df[value_col].to_numpy(np.float64)
    n = len(x)
    point = float(np.mean(x))

    boots = np.empty(reps, dtype=np.float64)
    for b in range(reps):
        idx = rng.integers(0, n, size=n)
        boots[b] = float(np.mean(x[idx]))

    lo, hi = np.quantile(boots, [0.025, 0.975])
    return point, float(lo), float(hi)

summary_rows = []
for mechanism in MECHANISMS:
    sub = query_mech[query_mech["mechanism"] == mechanism]
    for measure in MEASURES:
        point, lo, hi = bootstrap_mean_ci(sub, measure)
        summary_rows.append({
            "mechanism": mechanism,
            "measure": measure,
            "mean": point,
            "ci95_low": lo,
            "ci95_high": hi,
            "n_queries": len(sub),
        })

summary = pd.DataFrame(summary_rows)
SUMMARY_PATH = OUT / f"v023_{mode}_mechanism_component_summary.csv"
summary.to_csv(SUMMARY_PATH, index=False)

display(summary[
    summary["measure"].isin([
        "state_direct_norm",
        "state_prop_norm",
        "state_component_cosine",
        "state_cancellation_ratio",
        "candidate_direct_jaccard",
        "feedback_direct_cosdist",
        "utility_direct_abs",
    ])
].round(6))

In [ ]:
# Cell 14 — Round profiles with query-cluster uncertainty
round_rows = []

for mechanism in MECHANISMS:
    for t in range(ROUNDS):
        sub0 = query_mech_round[
            (query_mech_round["mechanism"] == mechanism)
            & (query_mech_round["round"] == t)
        ]
        for measure in [
            "state_direct_norm",
            "state_prop_norm",
            "state_component_cosine",
            "state_cancellation_ratio",
            "candidate_direct_jaccard",
            "candidate_prop_jaccard",
            "feedback_direct_cosdist",
            "feedback_prop_cosdist",
            "utility_direct_abs",
            "utility_prop_abs",
        ]:
            point, lo, hi = bootstrap_mean_ci(sub0, measure)
            round_rows.append({
                "mechanism": mechanism,
                "round": t,
                "measure": measure,
                "mean": point,
                "ci95_low": lo,
                "ci95_high": hi,
                "n_queries": len(sub0),
            })

round_summary = pd.DataFrame(round_rows)
ROUND_PATH = OUT / f"v023_{mode}_round_profiles.csv"
round_summary.to_csv(ROUND_PATH, index=False)

display(round_summary[
    round_summary["measure"].isin([
        "state_direct_norm",
        "state_prop_norm",
        "state_cancellation_ratio"
    ])
].round(6))

In [ ]:
# Cell 15 — Pairwise intervention differences at the QUERY level
# These are descriptive mechanism-class contrasts, not matched causal effects.

PAIRS = [
    ("e5_representation", "e5_nprobe"),
    ("e5_representation", "e5_hnsw"),
    ("e5_nprobe", "e5_hnsw"),
]

COMPARE_MEASURES = [
    "state_direct_norm",
    "state_prop_norm",
    "state_component_cosine",
    "state_cancellation_ratio",
    "candidate_direct_jaccard",
    "feedback_direct_cosdist",
    "utility_direct_abs",
]

wide = query_mech.pivot(index="query_id", columns="mechanism", values=COMPARE_MEASURES)

pair_rows = []
rng_pair = np.random.default_rng(SEED + 230)

for A, B in PAIRS:
    for measure in COMPARE_MEASURES:
        xa = wide[(measure, A)]
        xb = wide[(measure, B)]
        mask = xa.notna() & xb.notna()
        d = (xa[mask] - xb[mask]).to_numpy(np.float64)
        n = len(d)

        point = float(d.mean())
        boots = np.empty(BOOTSTRAP_REPS, dtype=np.float64)
        for b in range(BOOTSTRAP_REPS):
            idx = rng_pair.integers(0, n, size=n)
            boots[b] = float(d[idx].mean())

        lo, hi = np.quantile(boots, [0.025, 0.975])

        pair_rows.append({
            "A": A,
            "B": B,
            "measure": measure,
            "mean_A_minus_B": point,
            "ci95_low": float(lo),
            "ci95_high": float(hi),
            "n_queries": n,
        })

pairwise = pd.DataFrame(pair_rows)
PAIR_PATH = OUT / f"v023_{mode}_pairwise_query_differences.csv"
pairwise.to_csv(PAIR_PATH, index=False)

display(pairwise.round(6))

## Interpretation gate

The notebook does **not** automatically declare a “mechanism theorem.”

A stronger mechanism interpretation is justified only if the evidence consistently supports a distinction such as one or more of:

- representation approximation has a reliably different **fixed-common-state direct perturbation** profile;
- representation/search-effort differ in **propagated vs direct component balance**;
- their vector components show different **alignment/cancellation** profiles;
- the difference persists through candidate/feedback/utility proxies.

If those differences are weak, inconsistent across measures, or overlap heavily, the correct manuscript wording remains:

> **mechanism-dependent empirical regimes**

rather than a causal or operator-level explanation.

In [ ]:
# Cell 16 — Conservative evidence gate (descriptive, not a theorem)
def ci_excludes_zero(row):
    return (row["ci95_low"] > 0) or (row["ci95_high"] < 0)

# Count how many representation-vs-search-effort pairwise comparisons
# show a non-zero query-level mean difference for the preselected core measures.
core = pairwise[
    pairwise["A"].eq("e5_representation")
    & pairwise["B"].isin(["e5_nprobe","e5_hnsw"])
    & pairwise["measure"].isin([
        "state_direct_norm",
        "state_prop_norm",
        "state_component_cosine",
        "state_cancellation_ratio",
        "candidate_direct_jaccard",
        "feedback_direct_cosdist",
        "utility_direct_abs",
    ])
].copy()

core["ci_excludes_zero"] = core.apply(ci_excludes_zero, axis=1)

gate = {
    "status": "ARC_V023_COMPLETE",
    "mode": mode,
    "n_queries": int(len(selected_ids)),
    "core_rep_vs_search_effort_comparisons": int(len(core)),
    "core_nonzero_CI_count": int(core["ci_excludes_zero"].sum()),
    "interpretation": (
        "Inspect directions and round profiles before strengthening mechanism language. "
        "A high count alone is not sufficient: the pattern must be coherent across "
        "state/candidate/feedback/utility levels and must not be selectively reported."
    ),
    "test_accessed": False,
    "test_relevance_accessed": False,
}

display(core.round(6))
print(json.dumps(gate, indent=2))

In [ ]:
# Cell 17 — Final report + artifact hashes
REPORT = {
    **gate,
    "protocol_sha256": PROTOCOL_SHA,
    "interventions": PROTOCOL["interventions"],
    "source_hashes": {
        "v018_protocol": sha256_file(V018_PROTOCOL),
        "v013_split": sha256_file(V013_SPLIT),
        "hnsw_index": sha256_file(HNSW_INDEX),
        "hnsw_freeze": sha256_file(HNSW_FREEZE),
        "pq32_index": sha256_file(PQ32_PATH),
        "sq8_index": sha256_file(SQ8_PATH),
    },
    "primary_outputs": {
        "merged_replay": str(MERGED),
        "component_summary": str(SUMMARY_PATH),
        "round_profiles": str(ROUND_PATH),
        "pairwise_query_differences": str(PAIR_PATH),
    },
    "completed_at_utc": datetime.now(timezone.utc).isoformat(),
}

REPORT_PATH = OUT / f"v023_{mode}_final_report.json"
REPORT_PATH.write_text(json.dumps(REPORT, indent=2))

artifact_paths = [
    PROTOCOL_PATH,
    OUT / "v023_frozen_policy_grid.csv",
    MERGED,
    SUMMARY_PATH,
    ROUND_PATH,
    PAIR_PATH,
    REPORT_PATH,
]

hash_rows = []
for p in artifact_paths:
    if p.exists():
        hash_rows.append({
            "file": p.name,
            "bytes": p.stat().st_size,
            "sha256": sha256_file(p),
        })

hash_df = pd.DataFrame(hash_rows)
HASH_PATH = OUT / f"V023_{mode.upper()}_ARTIFACT_SHA256.csv"
hash_df.to_csv(HASH_PATH, index=False)

print("="*88)
print("ARC-v0.23 EQ.6 COMMON-STATE OPERATOR REPLAY — COMPLETE")
print("mode:", mode)
print("output:", OUT)
print("report:", REPORT_PATH)
print("test accessed:", False)
print("="*88)
display(hash_df)

## What to send back for review

After the full run, send either:

1. the executed notebook, or
2. these four outputs:
   - `v023_full_mechanism_component_summary.csv`
   - `v023_full_round_profiles.csv`
   - `v023_full_pairwise_query_differences.csv`
   - `v023_full_final_report.json`

The most important quantities for deciding manuscript language are:

- `state_direct_norm`
- `state_prop_norm`
- `state_component_cosine`
- `state_cancellation_ratio`
- `candidate_direct_jaccard`
- `feedback_direct_cosdist`
- `utility_direct_abs`

Do **not** tune the contrast, policy grid, or thresholds after seeing these outcomes.